In [11]:
# 최초 1회
# !pip install opencv-python numpy ipywidgets matplotlib
#
# Colab 확장 사용 시: 커널을 "Colab" 런타임으로 연결하면 cv2_imshow 자동 적용
# 로컬 Cursor/VS Code: matplotlib 밝기 보정 표시 (셀 1 재실행 필요)

In [12]:
import argparse
import json
import sys
import time
from pathlib import Path

import cv2
import numpy as np


def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        return shell in ("ZMQInteractiveShell", "Shell")
    except Exception:
        return False


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _resolve_display_backend():
    """Colab 런타임이면 cv2_imshow, 로컬이면 matplotlib(밝기 보정)."""
    if _in_colab():
        from google.colab.patches import cv2_imshow
        return "colab", cv2_imshow
    try:
        import matplotlib.pyplot as plt
        return "matplotlib", plt
    except ImportError:
        return "image", None


def _brighten_bgr(bgr, alpha=1.1, beta=20):
    """노트북 출력이 Colab보다 어둡게 보이는 현상 보정."""
    return cv2.convertScaleAbs(bgr, alpha=alpha, beta=beta)


def _show_frame(combined_bgr, frame_idx, blur, block, fft, backend=None):
    """Colab cv2_imshow 우선, 로컬은 matplotlib PNG."""
    if backend is None:
        backend, tool = _resolve_display_backend()
    else:
        _, tool = _resolve_display_backend()

    print(
        f"\n--- Frame {frame_idx} --- "
        f"Blur={blur:.1f} Block={block:.1f} FFT={fft:.2f}"
    )
    img = _brighten_bgr(combined_bgr)

    if backend == "colab":
        tool(img)
        return

    if backend == "matplotlib":
        import matplotlib.pyplot as plt
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(13, 3.8), dpi=110)
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")
        ax.imshow(rgb)
        ax.axis("off")
        ax.set_title(
            f"Frame {frame_idx}  |  Blur {blur:.1f}  Block {block:.1f}  FFT {fft:.2f}",
            fontsize=10,
            color="#222222",
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        return

    from IPython.display import Image, display
    ok, buf = cv2.imencode(".png", img)
    if ok:
        display(Image(data=buf.tobytes()))


def _make_overlay(frame_bgr, heatmap_bgr, frame_weight=0.55):
    """원본 비중을 높여 야간/저조도 영상도 덜 어둡게."""
    base = _brighten_bgr(frame_bgr, alpha=1.05, beta=10)
    heatmap_weight = 1.0 - frame_weight
    return cv2.addWeighted(base, frame_weight, heatmap_bgr, heatmap_weight, 0)

def calculate_blockiness_heatmap(gray_frame):
    """
    프레임 내의 8x8 픽셀 격자 경계에서 발생하는 압축 손실(단차)을 계산하여
    시각적인 히트맵(Heatmap) 형태로 반환합니다.
    """
    h, w = gray_frame.shape
    mask = np.zeros((h, w), dtype=np.float32)

    # 1. H.264/JPEG의 기본 압축 단위인 8x8 격자의 경계면 픽셀 차이 계산
    gray_float = gray_frame.astype(np.float32)

    # 정확한 8x8 경계선 인덱스 추출 (해상도에 구애받지 않도록 보정)
    h_bnd = np.arange(7, h - 1, 8)
    w_bnd = np.arange(7, w - 1, 8)

    # 수평/수직 경계선에서의 픽셀 단차(차이) 계산
    diff_h = np.abs(gray_float[h_bnd, :] - gray_float[h_bnd + 1, :])
    diff_v = np.abs(gray_float[:, w_bnd] - gray_float[:, w_bnd + 1])

    # 마스크에 단차 값 매핑
    mask[h_bnd, :] += diff_h
    mask[h_bnd + 1, :] += diff_h
    mask[:, w_bnd] += diff_v
    mask[:, w_bnd + 1] += diff_v

    # 2. 전체 픽셀 면적이 아닌 "순수 8x8 경계선 단차"만의 평균을 계산
    raw_score = (np.mean(diff_h) + np.mean(diff_v)) / 2.0

    # 직관적인 점수(0~100)를 위해 스케일링
    current_score = min(raw_score * 10.0, 100.0)

    # 3. 선으로만 되어있는 단차 값을 블록 전체로 부드럽게 퍼뜨림 (가우시안 블러)
    heatmap_blurred = cv2.GaussianBlur(mask, (15, 15), 0)

    # 4. 절대값 + 프레임 내 상대 정규화 혼합 (저압축 영상도 히트맵이 너무 까맣지 않게)
    heatmap_absolute = np.clip(heatmap_blurred * 10.0, 0, 255).astype(np.uint8)
    if float(heatmap_absolute.max()) < 48:
        heatmap_norm = cv2.normalize(heatmap_blurred, None, 0, 255, cv2.NORM_MINMAX)
        heatmap_absolute = np.maximum(
            heatmap_absolute, (heatmap_norm * 0.45).astype(np.uint8)
        )
    color_map = cv2.applyColorMap(heatmap_absolute, cv2.COLORMAP_JET)

    return color_map, current_score

def calculate_blur_score(gray_frame):
    """
    Laplacian 분산을 이용하여 프레임의 선명도(흐림 정도)를 계산합니다.
    값이 낮을수록 화면이 뭉개지고 흐리다는 것을 의미합니다.
    """
    return cv2.Laplacian(gray_frame, cv2.CV_64F).var()

def calculate_fft_grid_peak(gray_frame):
    """
    2D FFT를 통해 주파수 도메인 변환 후 이중 압축 힌트(격자 피크)를 계산합니다.
    """
    f_transform = np.fft.fft2(gray_frame)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)

    h, w = magnitude_spectrum.shape
    cy, cx = h // 2, w // 2

    # 중앙(저주파) 마스킹 처리
    mask_radius = 30
    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - cx)**2 + (Y - cy)**2)
    high_freq_region = magnitude_spectrum[dist_from_center > mask_radius]

    peak_strength = np.std(high_freq_region) / 100.0 if len(high_freq_region) > 0 else 0
    return min(float(peak_strength), 1.0)

def run_blockiness_viewer(
    video_path,
    *,
    sample_every=10,
    display_mode="auto",
    show_window=None,
    wait_ms=1,
    frame_delay_sec=0.05,
    overlay_frame_weight=0.55,
):
    """
    display_mode:
      - auto     : 노트북이면 notebook, 터미널이면 opencv
      - notebook : Colab처럼 셀에 프레임 이미지 출력 (권장)
      - opencv   : 별도 OpenCV 창
      - both     : 셀 + OpenCV 창
      - none     : 숫자 요약만
  """
    if show_window is not None:
        display_mode = "opencv" if show_window else "none"
    if display_mode == "auto":
        display_mode = "notebook" if _in_notebook() else "opencv"

    show_notebook = display_mode in ("notebook", "both")
    show_opencv = display_mode in ("opencv", "both")

    source = int(video_path) if str(video_path).isdigit() else str(video_path)
    cap = cv2.VideoCapture(source)

    if not cap.isOpened():
        print(f"'{video_path}' 영상을 열 수 없습니다.")
        return None

    print("프레임별 종합 화질/압축 컨텍스트 분석을 시작합니다...")
    if show_opencv:
        print("OpenCV 창: 'q' 키로 중단")
    if show_notebook:
        backend, _ = _resolve_display_backend()
        label = "Google Colab cv2_imshow" if backend == "colab" else f"로컬 {backend}"
        print(f"표시 방식: {label}")

    frame_idx = 0
    metrics_timeline = {"blur": [], "blockiness": [], "fft_peak": []}

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        if frame_idx % sample_every != 0:
            continue

        gray_original = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        heatmap_original, blockiness_score = calculate_blockiness_heatmap(gray_original)
        blur_score = calculate_blur_score(gray_original)
        fft_peak_score = calculate_fft_grid_peak(gray_original)

        metrics_timeline["blockiness"].append(blockiness_score)
        metrics_timeline["blur"].append(blur_score)
        metrics_timeline["fft_peak"].append(fft_peak_score)

        target_w, target_h = 480, 270
        frame_resized = cv2.resize(frame, (target_w, target_h))
        heatmap_resized = cv2.resize(heatmap_original, (target_w, target_h))
        overlay = _make_overlay(frame_resized, heatmap_resized, overlay_frame_weight)

        # 어두운 영상에서도 글자 보이게 검은 외곽선
        def _put_text(img, text, org, color):
            cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
            cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        _put_text(frame_resized, f"Frame: {frame_idx}", (10, 30), (255, 255, 255))
        _put_text(frame_resized, "[Metrics]", (10, 60), (0, 255, 0))

        blur_color = (0, 0, 255) if blur_score < 100 else (0, 255, 0)
        _put_text(frame_resized, f"Blur: {blur_score:.1f}", (10, 90), blur_color)

        fft_color = (0, 0, 255) if fft_peak_score > 0.4 else (0, 255, 255)
        _put_text(frame_resized, f"FFT Peak: {fft_peak_score:.2f}", (10, 120), fft_color)

        status_color = (0, 0, 255) if blockiness_score > 30 else (0, 255, 255)
        _put_text(overlay, f"Block Loss: {blockiness_score:.1f}", (10, 30), status_color)
        _put_text(overlay, "[Heatmap]", (10, 60), (0, 255, 255))

        combined_view = np.hstack((frame_resized, overlay))

        if show_notebook:
            _show_frame(
                combined_view, frame_idx, blur_score, blockiness_score, fft_peak_score
            )
            if frame_delay_sec > 0:
                time.sleep(frame_delay_sec)

        if show_opencv:
            cv2.imshow("Blockiness / Quality Context", combined_view)
            if cv2.waitKey(wait_ms) & 0xFF == ord("q"):
                print("사용자 중단 (q)")
                break
        elif display_mode == "none":
            print(
                f"Frame {frame_idx}: blur={blur_score:.1f} "
                f"block={blockiness_score:.1f} fft={fft_peak_score:.2f}"
            )

    cap.release()
    if show_opencv:
        cv2.destroyAllWindows()

    summary = build_summary(metrics_timeline, frame_idx)
    print_summary(summary)
    return summary


def build_summary(metrics_timeline, total_frames):
    if not metrics_timeline["blur"]:
        return {"samples": 0, "total_frames": total_frames}

    avg_blur = float(np.mean(metrics_timeline["blur"]))
    avg_block = float(np.mean(metrics_timeline["blockiness"]))
    avg_fft = float(np.mean(metrics_timeline["fft_peak"]))

    return {
        "total_frames": total_frames,
        "samples": len(metrics_timeline["blur"]),
        "quality_metrics": {
            "globalBlurScore": avg_blur,
            "globalBlockiness": round(avg_block / 100.0, 4),
            "fftGridPeakStrength": round(avg_fft, 4),
        },
        "interpretation": {
            "isHeavilyCompressed": avg_block >= 30 or avg_blur < 100,
            "recommendation": "NORMAL_PROCESS",
        },
    }


def print_summary(summary):
    print("\n=== 종합 분석 요약 ===")
    if not summary or summary.get("samples", 0) == 0:
        print("분석된 샘플이 없습니다.")
        return
    q = summary["quality_metrics"]
    block_raw = q["globalBlockiness"] * 100.0
    print(f"총 {summary['total_frames']} 프레임 중 {summary['samples']}개 샘플링")
    print(f"[Blur] 평균: {q['globalBlurScore']:.1f}")
    print(f"[Blockiness] 평균: {block_raw:.1f} (정규화 {q['globalBlockiness']:.4f})")
    print(f"[FFT Peak] 평균: {q['fftGridPeakStrength']:.2f}")


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="영상 화질/압축 컨텍스트 분석 (로컬)")
    parser.add_argument("video", nargs="?", default=None, help="영상 경로 또는 웹캠 번호(0)")
    parser.add_argument("--sample-every", type=int, default=10, help="N프레임마다 분석")
    parser.add_argument("--no-display", action="store_true", help="창 없이 콘솔만")
    parser.add_argument("--json-out", type=str, default=None, help="요약 JSON 저장 경로")
    return parser.parse_args(argv)

In [13]:
# Colab처럼 [영상 선택] 버튼 → [분석 시작]
import tempfile

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
    HAS_IPYWIDGETS = True
except ImportError:
    HAS_IPYWIDGETS = False
    print("ipywidgets 없음 → 터미널: pip install ipywidgets 후 커널 재시작")


def _read_upload(upload_widget):
    """ipywidgets 7(dict) / 8(tuple) 모두 지원."""
    val = upload_widget.value
    if not val:
        return None, None
    if isinstance(val, dict):
        name = next(iter(val))
        return name, bytes(val[name]["content"])
    info = val[0]
    return info["name"], bytes(info["content"])


def _save_temp_video(name: str, content: bytes) -> Path:
    suffix = Path(name).suffix or ".mp4"
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix, prefix="fs_upload_")
    tmp.write(content)
    tmp.close()
    return Path(tmp.name)


def pick_video_file():
    """노트북 UI 대신 Windows/Mac 파일 선택 창."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = filedialog.askopenfilename(
        title="분석할 영상 선택",
        filetypes=[
            ("Video", "*.mp4 *.avi *.mov *.mkv *.webm"),
            ("All files", "*.*"),
        ],
    )
    root.destroy()
    return path or None


if HAS_IPYWIDGETS:
    uploader = widgets.FileUpload(
        accept=".mp4,.avi,.mov,.mkv,.webm",
        multiple=False,
        description="영상 선택",
        button_style="primary",
    )
    run_btn = widgets.Button(description="분석 시작", button_style="success", icon="play")
    native_btn = widgets.Button(description="파일 찾기", icon="folder-open")
    sample_slider = widgets.IntSlider(value=10, min=1, max=30, description="N프레임마다")
    status = widgets.HTML("<i>① [영상 선택]으로 파일 업로드 → ② [분석 시작]</i>")
    out = widgets.Output()

    def _run_analysis(video_path: str, label: str):
        with out:
            clear_output(wait=True)
            print(f"분석 중: {label}")
            summary = run_blockiness_viewer(
                video_path,
                sample_every=sample_slider.value,
                display_mode="notebook",
                frame_delay_sec=0.05,
            )
            if summary:
                print("\n[JSON 요약]")
                display(summary)

    def on_upload_run(_):
        name, content = _read_upload(uploader)
        if not name:
            status.value = "<b style='color:red'>먼저 영상을 선택하세요.</b>"
            return
        temp_path = _save_temp_video(name, content)
        status.value = f"<b>업로드됨:</b> {name}"
        _run_analysis(str(temp_path), name)

    def on_native_pick(_):
        path = pick_video_file()
        if not path:
            status.value = "<i>파일 선택이 취소되었습니다.</i>"
            return
        status.value = f"<b>선택됨:</b> {path}"
        _run_analysis(path, path)

    run_btn.on_click(on_upload_run)
    native_btn.on_click(on_native_pick)

    display(
        widgets.VBox(
            [
                widgets.HTML("<h3>영상 화질/압축 컨텍스트 분석</h3>"),
                uploader,
                widgets.HBox([run_btn, native_btn]),
                sample_slider,
                status,
                out,
            ]
        )
    )
else:
    path = pick_video_file()
    if path:
        summary = run_blockiness_viewer(path, sample_every=10, display_mode="notebook")
        summary